# Offline Policy Comparison

This notebook compares the popularity baseline against the selected hybrid recommender using paired user-level outcomes. This is not a real online A/B test but simply an offline policy comparison using held-out future ratings.

## Design

Control policy: popularity recommender.

Treatment policy: three-signal hybrid recommender with 50% popularity, 30% content similarity, and 20% genre preference.

Primary offline decision metrics: rated Precision@10 and catalog coverage. Standard ranking metrics are included as secondary diagnostics because MovieLens does not include impression logs.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

for path in [current_path, *current_path.parents]:
    if (path / "src" / "data.py").exists():
        project_root = path
        break
else:
    raise FileNotFoundError("Could not find project root containing src/data.py")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd

from src.content import build_movie_content_features
from src.data import load_movies
from src.experiments import summarize_paired_difference
from src.metrics import precision_at_k, recall_at_k, ndcg_at_k, rated_precision_at_k
from src.recommenders import (
    build_content_tfidf_matrix,
    recommend_popular_movies,
    recommend_true_hybrid_movies_from_vectors,
)

train = pd.read_csv(project_root / "data/processed/train_ratings.csv")
test = pd.read_csv(project_root / "data/processed/test_ratings.csv")
movies = load_movies()
tmdb_metadata = pd.read_csv(project_root / "data/processed/tmdb_movie_metadata.csv")

movie_content = build_movie_content_features(
    movies=movies,
    tmdb_metadata=tmdb_metadata,
)

_, content_vectors = build_content_tfidf_matrix(
    movie_content,
    text_column="overview_genre_text",
)

## Build User-Level Policy Outcomes

For each user, generate top-10 recommendations from both policies and evaluate them against the user's held-out ratings.

In [2]:
def calculate_user_policy_outcome(user_id: int, policy_name: str, recs: pd.DataFrame, k: int = 10) -> dict:
    recommended_items = recs["movieId"].tolist()

    user_test = test[test["userId"] == user_id]
    rated_items = set(user_test["movieId"])
    positive_items = set(user_test[user_test["is_positive"]]["movieId"])

    top_k_items = recommended_items[:k]
    rated_recommendations = [item for item in top_k_items if item in rated_items]
    positive_hits = [item for item in top_k_items if item in positive_items]

    return {
        "userId": user_id,
        "policy": policy_name,
        "precision_at_10": precision_at_k(
            recommended_items=recommended_items,
            relevant_items=positive_items,
            k=k,
        ),
        "recall_at_10": recall_at_k(
            recommended_items=recommended_items,
            relevant_items=positive_items,
            k=k,
        ),
        "ndcg_at_10": ndcg_at_k(
            recommended_items=recommended_items,
            relevant_items=positive_items,
            k=k,
        ),
        "rated_precision_at_10": rated_precision_at_k(
            recommended_items=recommended_items,
            positive_items=positive_items,
            rated_items=rated_items,
            k=k,
        ),
        "positive_hits_at_10": len(positive_hits),
        "rated_recommendations_at_10": len(rated_recommendations),
    }


policy_rows = []
recommendation_rows = []

for user_id in test["userId"].unique():
    control_recs = recommend_popular_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=10,
    ).copy()

    treatment_recs = recommend_true_hybrid_movies_from_vectors(
        train_ratings=train,
        movie_content=movie_content,
        content_vectors=content_vectors,
        user_id=user_id,
        k=10,
        popularity_weight=0.50,
        content_weight=0.30,
        genre_weight=0.20,
    ).copy()

    policy_rows.append(
        calculate_user_policy_outcome(
            user_id=user_id,
            policy_name="control_popularity",
            recs=control_recs,
        )
    )

    policy_rows.append(
        calculate_user_policy_outcome(
            user_id=user_id,
            policy_name="treatment_hybrid_50_30_20",
            recs=treatment_recs,
        )
    )

    control_recs["userId"] = user_id
    control_recs["policy"] = "control_popularity"
    control_recs["rank"] = range(1, len(control_recs) + 1)

    treatment_recs["userId"] = user_id
    treatment_recs["policy"] = "treatment_hybrid_50_30_20"
    treatment_recs["rank"] = range(1, len(treatment_recs) + 1)

    recommendation_rows.extend([control_recs, treatment_recs])

policy_outcomes = pd.DataFrame(policy_rows)
policy_recommendations = pd.concat(recommendation_rows, ignore_index=True)

policy_outcomes.head()

,userId,policy,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,positive_hits_at_10,rated_recommendations_at_10
0,1,control_popularity,0.3,0.078947,0.315163,1.0,3,3
1,1,treatment_hybrid_50_30_20,0.1,0.026316,0.085143,1.0,1,1
2,2,control_popularity,0.0,0.000000,0.000000,NaN,0,0
3,2,treatment_hybrid_50_30_20,0.0,0.000000,0.000000,NaN,0,0
4,3,control_popularity,0.0,0.000000,0.000000,NaN,0,0


## Aggregate Policy Metrics

Summarize user-level outcomes for each policy.

In [3]:
policy_summary = (
    policy_outcomes
    .groupby("policy")
    .agg(
        users=("userId", "nunique"),
        precision_at_10=("precision_at_10", "mean"),
        recall_at_10=("recall_at_10", "mean"),
        ndcg_at_10=("ndcg_at_10", "mean"),
        rated_precision_at_10=("rated_precision_at_10", "mean"),
        users_with_rated_recommendations=("rated_precision_at_10", "count"),
        positive_hits_at_10=("positive_hits_at_10", "mean"),
        rated_recommendations_at_10=("rated_recommendations_at_10", "mean"),
    )
    .reset_index()
)

policy_summary

,policy,users,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,positive_hits_at_10,rated_recommendations_at_10
0,control_popularity,610,0.057869,0.049672,0.075906,0.771876,221,0.578689,0.745902
1,treatment_hybrid_50_30_20,610,0.042459,0.035851,0.053543,0.780215,210,0.424590,0.555738


## Catalog Diagnostics

Compare how broad each policy is across the full user base.

In [4]:
positive_counts = (
    train[train["is_positive"]]
    .groupby("movieId")
    .size()
    .reset_index(name="global_positive_ratings")
)

policy_recommendations = policy_recommendations.merge(
    positive_counts,
    on="movieId",
    how="left",
)

policy_recommendations["global_positive_ratings"] = (
    policy_recommendations["global_positive_ratings"].fillna(0)
)

catalog_summary = (
    policy_recommendations
    .groupby("policy")
    .agg(
        unique_recommended_movies=("movieId", "nunique"),
        avg_global_positive_ratings=("global_positive_ratings", "mean"),
        median_global_positive_ratings=("global_positive_ratings", "median"),
    )
    .reset_index()
)

catalog_summary["catalog_coverage"] = (
    catalog_summary["unique_recommended_movies"] / movies["movieId"].nunique()
)

catalog_summary

,policy,unique_recommended_movies,avg_global_positive_ratings,median_global_positive_ratings,catalog_coverage
0,control_popularity,91,165.132131,155.0,0.009341
1,treatment_hybrid_50_30_20,508,92.660328,71.0,0.052145


## Paired User-Level Differences

Because every user is evaluated under both policies offline, compare treatment minus control within each user.

In [5]:
paired_outcomes = policy_outcomes.pivot(
    index="userId",
    columns="policy",
    values=[
        "rated_precision_at_10",
        "positive_hits_at_10",
        "precision_at_10",
        "recall_at_10",
        "ndcg_at_10",
    ],
)

paired_outcomes.columns = [
    f"{metric}_{policy}"
    for metric, policy in paired_outcomes.columns
]

paired_outcomes = paired_outcomes.reset_index()

paired_summaries = pd.DataFrame(
    [
        {
            "metric": "rated_precision_at_10",
            **summarize_paired_difference(
                data=paired_outcomes,
                control_column="rated_precision_at_10_control_popularity",
                treatment_column="rated_precision_at_10_treatment_hybrid_50_30_20",
            ),
        },
        {
            "metric": "positive_hits_at_10",
            **summarize_paired_difference(
                data=paired_outcomes,
                control_column="positive_hits_at_10_control_popularity",
                treatment_column="positive_hits_at_10_treatment_hybrid_50_30_20",
            ),
        },
        {
            "metric": "precision_at_10",
            **summarize_paired_difference(
                data=paired_outcomes,
                control_column="precision_at_10_control_popularity",
                treatment_column="precision_at_10_treatment_hybrid_50_30_20",
            ),
        },
        {
            "metric": "ndcg_at_10",
            **summarize_paired_difference(
                data=paired_outcomes,
                control_column="ndcg_at_10_control_popularity",
                treatment_column="ndcg_at_10_treatment_hybrid_50_30_20",
            ),
        },
    ]
)

paired_summaries

,metric,n_users,control_mean,treatment_mean,mean_difference,ci_lower,ci_upper,treatment_win_rate,control_win_rate,tie_rate
0,rated_precision_at_10,153,0.757633,0.798553,0.040920,-0.018795,0.100635,0.228758,0.143791,0.627451
1,positive_hits_at_10,610,0.578689,0.424590,-0.154098,-0.225392,-0.082805,0.122951,0.198361,0.678689
2,precision_at_10,610,0.057869,0.042459,-0.015410,-0.022539,-0.008280,0.122951,0.198361,0.678689
3,ndcg_at_10,610,0.075906,0.053543,-0.022363,-0.032432,-0.012294,0.159016,0.237705,0.603279


## Decision Framing

The treatment should not be interpreted as proven causal lift. The dataset has no impression logs, so missing ratings are not dislikes. This analysis estimates whether the selected hybrid is a credible candidate for an online test.

In [6]:
decision_summary = pd.DataFrame(
    {
        "decision_question": ["Should the hybrid move forward as an online test candidate?"],
        "offline_recommendation": ["Yes, if the product goal values catalog discovery and rated-only relevance."],
        "main_reason": ["The hybrid increases catalog coverage and rated-only precision, while standard ranking metrics remain secondary diagnostics due to exposure bias."],
        "key_limitation": ["This is offline policy evaluation, not a randomized online experiment."],
    }
)

decision_summary

,decision_question,offline_recommendation,main_reason,key_limitation
0,Should the hybrid move forward as an online te...,"Yes, if the product goal values catalog discov...",The hybrid increases catalog coverage and rate...,"This is offline policy evaluation, not a rando..."
